In [0]:
dbutils.widgets.text("source_system", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("source_file_format", "")

source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")


In [0]:
landing_path = f"/Volumes/adb_caresync/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/"

# Check if the landing path (or any parent directory) exists
try:
    dbutils.fs.ls(landing_path)
    path_exists = True
except Exception:
    path_exists = False

if not path_exists:
    print(f"No new files received. Path does not exist: {landing_path}")
    dbutils.notebook.exit("No new files received")

In [0]:
if source_file_format.lower() == "csv":
    landing_df = spark.read.format("csv").option("header", True).option("inferSchema", True).load(landing_path)
else:
    landing_df = spark.read.format("parquet").load(landing_path)

display(landing_df)

In [0]:
#add 2 columns in landing_df, 1. landing_timestamp 2. insert_timestamp as current timestamp

from pyspark.sql.functions import current_timestamp, lit

updated_df = (
    landing_df
    .withColumn("landing_timestamp", lit(landing_timestamp))
    .withColumn("insert_timestamp", current_timestamp())
)


In [0]:
display(updated_df)

In [0]:
# 1. step
updated_df.write.mode("append").saveAsTable(f"adb_caresync.bronze.{table_name}")

In [0]:
%sql
--to check whether the bronze table is created
select * from adb_caresync.bronze.hospitals